
### Low-ℓ BB — Full-Bundle Coadd Tile Viewer


#### Workflow
 --------
 1. Set paths and configuration (§1)
 2. Load coadd T / Q / U from FITS (§2)
 3. Load available apodisation + point-source masks 
 4. Inspect field extent from observed pixels 
 5. Build tile grid 
 6. Tile loop — **unmasked** first, to see raw point sources 
 7. Tile loop — **with mask applied** 

In [1]:
import os
import sys
import numpy as np
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt
from spt3g import core, maps
from astropy.io import fits
from matplotlib.backends.backend_pdf import PdfPages


In [7]:
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, _here)

from Plot import (
    apply_spt_style,
    mask_map,
    show_map_full_field,
    show_map_thumbnail,
)

apply_spt_style()

In [8]:
# Coadd path 
# "/sptgrid/analysis/spt3g_d1_midell_tqu_healpix/real_data_maps/full/full_220ghz.fits" 
DATA_DIR   = "/sptgrid/analysis/spt3g_d1_midell_tqu_healpix"  

# print the content of the data directory to verify the path
print("Contents of data directory:")
for item in os.listdir(DATA_DIR):
    print(item)


COADD_FILE  = "real_data_maps/full"   
COADD_PATH = os.path.join(DATA_DIR, COADD_FILE)

# Print the name of the coadd fits inside the path,
print("\nCoadd files in the coadd path:")
for item in os.listdir(COADD_PATH):
        print(item)

full_220ghz = COADD_PATH + "/full_220ghz.fits"
full_150ghz = COADD_PATH + "/full_150ghz.fits"
full_095ghz  = COADD_PATH + "/full_095ghz.fits"


# ── Mask directory ────────────────────────────────────────────────────────────
MASK_DIR = "/sptlocal/user/creichardt/bb2020"

# Print the content of the mask directory 
print("\nContents of mask directory:")
for item in os.listdir(MASK_DIR):
    print(item)




Contents of data directory:
README.html
simulated_maps
ancillary_products
real_data_maps

Coadd files in the coadd path:
full_220ghz.fits
full_150ghz.fits
full_095ghz.fits

Contents of mask directory:
puremask_0p5medwt_100mJy_30arcmin.npz
c2mask_0p5medwt_120arcmin.npz
circletest_puremask_0p5medwt_radius480_240arcmin.npz
circletest_c2puremask_0p5medwt_radius960_120arcmin.npz
puremask_0p5medwt_250mJy_180arcmin.npz
circletest_puremask_0p5medwt_radius960_480arcmin.npz
border_lines_0p5medwt.npz
circletest_puremask_0p5medwt_radius1800_240arcmin.npz
circletest_c2puremask_0p5medwt_radius2400_240arcmin.npz
distance_xy_0p5medwt.npz
distance_0p5medwt.npz
mask_badpixels_180arcmin.npz
circletest_c2puremask_0p5medwt_radius960_240arcmin.npz
puremask_0p5medwt_60arcmin.npz
circletest_puremask_0p5medwt_radius480_120arcmin.npz
bk8192_250mJy_nodisk20.npz
circletest_c2puremask_0p5medwt_radius2400_120arcmin.npz
circletest_puremask_0p5medwt_radius2400_480arcmin.npz
distance8192_0p6deg_xy_0p5medwt.npz
circlet

In [9]:
MASK_FILES = {
    "mask_250_30"   : "puremask8192_0p5medwt_250mJy_30arcmin.npz",
    "mask_250_60"   : "puremask8192_0p5medwt_250mJy_60arcmin.npz",
    "mask_250_nd30" : "puremask8192_0p5medwt_250mJy_nodisk_30arcmin.npz",
    "mask_250_nd60" : "puremask8192_0p5medwt_250mJy_nodisk_60arcmin.npz",
    "mask_100_30"   : "puremask8192_0p5medwt_100mJy_30arcmin.npz",
    "mask_apod_30"  : "puremask8192_0p5medwt_30arcmin.npz",
    "mask_apod_60"  : "puremask8192_0p5medwt_60arcmin.npz",
}

In [10]:
# display parameters 
RESO_ARCMIN  = 0.5     # arcmin per pixel  
PATCH_DEG    = 15.0     # square patch width = height in degrees
PATCH_PIX    = int(PATCH_DEG * 60 / RESO_ARCMIN)   # = 900 px for 3° patches

STOKES_KEYS  = ["T", "Q", "U"]
CMAP         = "coolwarm"

# Set to a directory path to save PDFs instead of displaying inline.

SAVE_DIR = "/sptlocal/user/vwelke/lowl_bb_tiles"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Patch size  : {PATCH_DEG}° × {PATCH_DEG}°  =  {PATCH_PIX} × {PATCH_PIX} px")
print(f"Resolution  : {RESO_ARCMIN} arcmin/px")

Patch size  : 15.0° × 15.0°  =  1800 × 1800 px
Resolution  : 0.5 arcmin/px


In [11]:
# inspect fits header to check number of fields

fits_file = full_220ghz

with fits.open(fits_file) as hdul:
    hdul.info()  # shows HDUs and table names

    # Usually the HEALPix map table is HDU 1
    hdr = hdul[1].header
    print("\n--- Header cards related to columns ---")
    for k, v in hdr.items():
        if str(k).startswith("TTYPE") or str(k).startswith("TFORM"):
            print(f"{k:8s} = {v}")

    print("\n--- Explicit column list ---")
    for i, col in enumerate(hdul[1].columns):
        print(f"field={i}: name={col.name}, format={col.format}")

Filename: /sptgrid/analysis/spt3g_d1_midell_tqu_healpix/real_data_maps/full/full_220ghz.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     69   36557509R x 10C   [J, D, D, D, D, D, D, D, D, D]   

--- Header cards related to columns ---
TTYPE1   = PIXEL
TFORM1   = J
TTYPE2   = TEMPERATURE
TFORM2   = D
TTYPE3   = Q_POLARISATION
TFORM3   = D
TTYPE4   = U_POLARISATION
TFORM4   = D
TTYPE5   = TT_NORMALIZED
TFORM5   = D
TTYPE6   = TQ_NORMALIZED
TFORM6   = D
TTYPE7   = TU_NORMALIZED
TFORM7   = D
TTYPE8   = QQ_NORMALIZED
TFORM8   = D
TTYPE9   = QU_NORMALIZED
TFORM9   = D
TTYPE10  = UU_NORMALIZED
TFORM10  = D

--- Explicit column list ---
field=0: name=PIXEL, format=J
field=1: name=TEMPERATURE, format=D
field=2: name=Q_POLARISATION, format=D
field=3: name=U_POLARISATION, format=D
field=4: name=TT_NORMALIZED, format=D
field=5: name=TQ_NORMALIZED, format=D
field=6: name=TU_NORMALIZED, format=D

In [ ]:
# 220 ghz map
T_c, Q_c, U_c = hp.read_map(full_220ghz, field=(0, 1, 2), partial=False)


In [ ]:
# inspect header
with fits.open(full_220ghz) as hdul:
    hdul.info()
    print("\nPrimary header:")
    print(repr(hdul[0].header))

    if len(hdul) > 1:
        print("\nMap HDU header:")
        print(repr(hdul[1].header))

Filename: /sptgrid/analysis/spt3g_d1_midell_tqu_healpix/real_data_maps/full/full_220ghz.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     69   36557509R x 10C   [J, D, D, D, D, D, D, D, D, D]   

Primary header:
SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                    8 / array data type                                
NAXIS   =                    0 / number of array dimensions                     
EXTEND  =                    T                                                  

Map HDU header:
XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                   76 / length of dimension 1                          
NAXIS2  =         

In [ ]:
# Build the observed-pixel boolean mask

obs_mask = (
        np.isfinite(T_c) & (T_c != hp.UNSEEN) &
        np.isfinite(Q_c) & (Q_c != hp.UNSEEN) &
        np.isfinite(U_c) & (U_c != hp.UNSEEN)
    )


In [ ]:
# Create a dictionary to hold the Stokes maps for easy access
stokes_maps = {"T": T_c, "Q": Q_c, "U": U_c}
nside_c     = hp.get_nside(T_c) 

print(f"nside            : {nside_c}")
# this is the number of pixels in the map that are observed (not UNSEEN or NaN) 
print(f"Observed pixels  : {obs_mask.sum():,}  ({obs_mask.sum()/len(T_c)*100:.2f}% of sky)")

nside            : 8192


Observed pixels  : 36,557,509  (4.54% of sky)


In [ ]:
# load one of the masks
MASK_PATH = os.path.join(MASK_DIR, "puremask8192_0p5medwt_250mJy_30arcmin.npz")
data      = np.load(MASK_PATH)
# this 
apod      = data[data.files[0]].astype(float)


print(f"Mask loaded, shape: {apod.shape}, key: '{data.files[0]}'")

Mask loaded, shape: (805306368,), key: 'mask'


In [ ]:
# Check the angular span of the observed region by converting pixel indices to RA/Dec coordinates
obs_pix        = np.where(obs_mask)[0]

# given resolution and pixel index, return the sky position
theta_c, phi_c = hp.pix2ang(nside_c, obs_pix)

ra_obs  = np.degrees(phi_c)
# Convert RA to range -180 to +180 degrees, default is 0 to 360 degrees
ra_obs  = np.where(ra_obs > 180, ra_obs - 360, ra_obs)   
dec_obs = 90.0 - np.degrees(theta_c)

ra_min,  ra_max  = ra_obs.min(),  ra_obs.max()
dec_min, dec_max = dec_obs.min(), dec_obs.max()

print(f"RA  range  :  {ra_min:.2f}° → {ra_max:.2f}°   span = {ra_max-ra_min:.2f}°")
print(f"Dec range  : {dec_min:.2f}° → {dec_max:.2f}°   span = {dec_max-dec_min:.2f}°")


RA  range  :  -56.51° → 53.76°   span = 110.28°
Dec range  : -71.98° → -40.20°   span = 31.78°


In [ ]:
# Quick full-field overview 

# mask_map applies the mask to the map, setting unobserved pixels to hp.UNSEEN
m_T_full = mask_map(U_c, obs_mask.astype(float))

rms_T    = float(np.std(m_T_full[m_T_full != hp.UNSEEN]))

#show_map_full_field(
#    m_T_full, vmin=-0.25*rms_T, vmax=0.25*rms_T,
#    title="Full-bundle coadd — U  (full field overview)",
#    unit="Tcmb", cmap=CMAP,
#)

In [ ]:
# Define the centre of patches , path deg was 3 degress
ra_centres  = np.arange(ra_min  + PATCH_DEG/2, ra_max  + PATCH_DEG/2, PATCH_DEG)
dec_centres = np.arange(dec_min + PATCH_DEG/2, dec_max + PATCH_DEG/2, PATCH_DEG)

n_ra, n_dec = len(ra_centres), len(dec_centres)
n_total     = n_ra * n_dec * len(STOKES_KEYS)  

# Print number of centers and the ranges
print(f"RA  centres  : {n_ra}  ({ra_centres[0]:.1f}° … {ra_centres[-1]:.1f}°)")
print(f"Dec centres  : {n_dec}  ({dec_centres[0]:.1f}° … {dec_centres[-1]:.1f}°)")
print(f"Patches/Stokes : {n_ra * n_dec}")
print(f"Total panels : {n_total}  ({len(STOKES_KEYS)} Stokes × {n_ra*n_dec} patches)")


RA  centres  : 8  (-49.0° … 56.0°)
Dec centres  : 3  (-64.5° … -34.5°)
Patches/Stokes : 24
Total panels : 72  (3 Stokes × 24 patches)


In [ ]:
def _tile_loop(stokes_maps_in, obs_mask_in, label="", sub_dir="", pdf_stokes="U", progress_every=5):
    """
    Iterate over all tiles and Stokes components.

    If SAVE_DIR is set, save all patches for one Stokes component (pdf_stokes)
    into a single multi-page PDF. If SAVE_DIR is None, show patches inline.
    Prints elapsed time and ETA so progress is visible during long runs.
    """
    import time, os, psutil
    p = psutil.Process(os.getpid())

    t0_all = time.perf_counter()

    if SAVE_DIR is not None:
        os.makedirs(SAVE_DIR, exist_ok=True)
        if sub_dir:
            os.makedirs(os.path.join(SAVE_DIR, sub_dir), exist_ok=True)

    tiles_per_stokes = len(ra_centres) * len(dec_centres)

    for stokes in STOKES_KEYS:
        t0_stokes = time.perf_counter()

        arr = stokes_maps_in[stokes].copy()
        arr[~obs_mask_in] = hp.UNSEEN

        rms = float(np.std(arr[obs_mask_in]))
        vmin, vmax = -1 * rms, 1 * rms

        print(f"[{stokes}] start | rms={rms:.4g} | tiles={tiles_per_stokes}")

        if SAVE_DIR is not None:
            if stokes != pdf_stokes:
                print(f"[{stokes}] skipped (saving only {pdf_stokes} to PDF)")
                continue

            safe_label = label.replace(" ", "_").replace("/", "-")
            pdf_name = f"tiles_{stokes}{('_' + safe_label) if safe_label else ''}.pdf"
            pdf_path = os.path.join(SAVE_DIR, sub_dir, pdf_name) if sub_dir else os.path.join(SAVE_DIR, pdf_name)

            tile_index = 0
            with PdfPages(pdf_path) as pdf:
                for dec_c in dec_centres:
                    for ra_c in ra_centres:
                        tile_index += 1

                        if (
                            tile_index == 1
                            or tile_index == tiles_per_stokes
                            or tile_index % progress_every == 0
                        ):
                            elapsed = time.perf_counter() - t0_stokes
                            rate = tile_index / elapsed if elapsed > 0 else 0.0
                            eta = (tiles_per_stokes - tile_index) / rate if rate > 0 else float("inf")
                            print(
                                f"[{stokes}] {tile_index}/{tiles_per_stokes} "
                                f"| elapsed={elapsed:.1f}s | ETA={eta:.1f}s"
                            )

                        title = (
                            f"Tile {tile_index}/{tiles_per_stokes} - "
                            f"{stokes}{('  ' + label) if label else ''}  |  "
                            f"RA {ra_c:+.1f} deg  Dec {dec_c:+.1f} deg"
                        )
                        rot = (ra_c, dec_c, 0)

                        fig = plt.figure(figsize=(6, 6), facecolor="white")
                        hp.gnomview(
                            arr,
                            rot=rot,
                            xsize=PATCH_PIX, ysize=PATCH_PIX,
                            reso=RESO_ARCMIN,
                            cmap=CMAP, min=vmin, max=vmax,
                            badcolor="white",
                            title=title, unit="Tcmb",
                            fig=fig.number,
                        )
                        pdf.savefig(fig, bbox_inches="tight", dpi=120)
                        plt.close(fig)

            dt = time.perf_counter() - t0_stokes
            print(f"[{stokes}] done in {dt:.1f}s | saved: {pdf_path}")

        else:
            tile_index = 0
            for dec_c in dec_centres:
                for ra_c in ra_centres:
                    tile_index += 1

                    if (
                        tile_index == 1
                        or tile_index == tiles_per_stokes
                        or tile_index % progress_every == 0
                    ):
                        elapsed = time.perf_counter() - t0_stokes
                        rate = tile_index / elapsed if elapsed > 0 else 0.0
                        eta = (tiles_per_stokes - tile_index) / rate if rate > 0 else float("inf")
                        print(
                            f"[{stokes}] {tile_index}/{tiles_per_stokes} "
                            f"| elapsed={elapsed:.1f}s | ETA={eta:.1f}s"
                        )

                    title = (
                        f"Tile {tile_index}/{tiles_per_stokes} - "
                        f"{stokes}{('  ' + label) if label else ''}  |  "
                        f"RA {ra_c:+.1f} deg  Dec {dec_c:+.1f} deg"
                    )
                    rot = (ra_c, dec_c, 0)

                    show_map_thumbnail(
                        arr, vmin=vmin, vmax=vmax,
                        title=title, unit="Tcmb",
                        cmap=CMAP,
                        rot=rot,
                        xsize=PATCH_PIX, ysize=PATCH_PIX,
                        reso=RESO_ARCMIN,
                    )

            dt = time.perf_counter() - t0_stokes
            rss_gb = p.memory_info().rss / (1024**3)
            print(f"[health] RAM={rss_gb:.2f} GB | threads={p.num_threads()}")
            print(f"[{stokes}] done in {dt:.1f}s (inline mode)")

    print(f"All done in {time.perf_counter() - t0_all:.1f}s")

In [ ]:
# Plot 



#print("=== Unmasked tiles ===")
#_tile_loop(stokes_maps, obs_mask, label="unmasked")

In [ ]:
import os, subprocess, getpass, psutil

user = getpass.getuser()
cmd = f"pgrep -u {user} -fa 'jupyter.*kernel|ipykernel' | wc -l"
n_kernels = int(subprocess.check_output(cmd, shell=True).decode().strip())

p = psutil.Process(os.getpid())
rss_gb = p.memory_info().rss / (1024**3)

print(f"Running kernels: {n_kernels}")
print(f"This kernel PID: {p.pid}")
print(f"This kernel RAM: {rss_gb:.2f} GB")

Running kernels: 54
This kernel PID: 1225175
This kernel RAM: 32.29 GB


In [ ]:
ACTIVE_MASK = "mask_250_nd30"   # key in MASK_FILES

# Load the actual mask array
mask_file = MASK_FILES[ACTIVE_MASK]
mask_path = os.path.join(MASK_DIR, mask_file)
with np.load(mask_path) as d:
    apod = d[d.files[0]].astype(float)
    mask_key = d.files[0]

# Optional nside check
nside_mask = hp.get_nside(apod)
if nside_mask != nside_c:
    print(f"Warning: mask nside={nside_mask}, map nside={nside_c} - upgrading mask.")
    apod = hp.ud_grade(apod, nside_out=nside_c)

# Combined observed mask
obs_and_mask = obs_mask & (apod > 0)

# Apply apodization
stokes_masked = {}
for stokes in STOKES_KEYS:
    arr = stokes_maps[stokes].copy()
    arr[obs_and_mask] *= apod[obs_and_mask]
    arr[~obs_and_mask] = hp.UNSEEN
    stokes_masked[stokes] = arr

print(f"Active mask: {ACTIVE_MASK} ({mask_file}), key='{mask_key}', shape={apod.shape}")
print(f"Pixels after mask: {obs_and_mask.sum():,} (was {obs_mask.sum():,} unmasked)")


NameError: name 'MASK_FILES' is not defined

In [ ]:
# Just oen patch for testing the maps
# maybe plot them 1 by 1 , and then plt.close() after each one, to avoid memory issues with too many open figures


stokes = "U"
ra_c = ra_centres[0]
dec_c = dec_centres[0]

arr = stokes_masked[stokes].copy()
arr[~obs_and_mask] = hp.UNSEEN

rms = float(np.std(arr[obs_and_mask]))
vmin, vmax = -rms, rms

show_map_thumbnail(
    arr,
    vmin=vmin,
    vmax=vmax,
    title=f"{stokes} | RA {ra_c:+.1f} deg  Dec {dec_c:+.1f} deg",
    unit="Tcmb",
    cmap=CMAP,
    rot=(ra_c, dec_c, 0),
    xsize=PATCH_PIX,
    ysize=PATCH_PIX,
    reso=RESO_ARCMIN,
)
# Maybe can save the image to disk later on, so can refer back to it 
#plt.close()

NameError: name 'ra_centres' is not defined

In [ ]:
print(f"\n=== Masked tiles [{ACTIVE_MASK}] ===")
for s in ["T"]:
    _tile_loop(
        stokes_masked,
        obs_and_mask,
        label=ACTIVE_MASK,
        sub_dir=ACTIVE_MASK,
        pdf_stokes=s,
        progress_every=10,
    )


=== Masked tiles [mask_250_nd30] ===


### Angular Power Spectrum

In [ ]:
LMAX = 1024
cl = hp.anafast(wmap_map_I_masked.filled(), lmax=LMAX)
ell = np.arange(len(cl))

In [ ]:
# Plot a normalised spectrum and write it to disk 
plt.figure(figsize=(10, 5))
plt.plot(ell, ell * (ell + 1) * cl)
plt.xlabel("$\ell$")
plt.ylabel("$\ell(\ell+1)C_{\ell}$")
plt.grid()
hp.write_cl("cl.fits", cl, overwrite=True)